# 🛡️ Phish-Guard — ML Model Analysis & Statistics
**Dataset:** PhiUSIIL Phishing URL Dataset  
**Model:** TF-IDF (char n-grams) + Random Forest Classifier  

This notebook covers:
- Dataset exploration & class distribution
- TF-IDF feature analysis
- Model training & evaluation
- Confusion Matrix
- Accuracy, Precision, Recall, F1-Score
- ROC Curve & AUC
- Feature importance (top TF-IDF characters)
- Cross-validation results

In [ ]:
# ─── Imports ───────────────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_score, recall_score, f1_score, accuracy_score
)

# ─── Plot Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.family':      'DejaVu Sans',
})

PHISH_COLOR  = '#f85149'   # red
LEGIT_COLOR  = '#3fb950'   # green
ACCENT_COLOR = '#58a6ff'   # blue

print('✅ Libraries loaded')

## 1. Load & Explore the Dataset

In [ ]:
# ─── Load Dataset ───────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATASET_PATH = os.path.join(NOTEBOOK_DIR, 'datasets', 'PhiUSIIL_Phishing_URL_Dataset.csv')

assert os.path.exists(DATASET_PATH), f'Dataset not found at: {DATASET_PATH}'

df_raw = pd.read_csv(DATASET_PATH)
print(f'📊 Raw shape: {df_raw.shape}')
print(f'📋 Columns ({len(df_raw.columns)}): {list(df_raw.columns)}')
df_raw.head(3)

In [ ]:
# ─── Column Detection & Preprocessing ──────────────────────────────────────────
cols_lower = {c.lower(): c for c in df_raw.columns}

url_col   = cols_lower.get('url') or cols_lower.get('domain')
label_col = cols_lower.get('label') or cols_lower.get('type') or cols_lower.get('status')

print(f'✅ URL column    : "{url_col}"')
print(f'✅ Label column  : "{label_col}"')

df = df_raw.rename(columns={url_col: 'url', label_col: 'label'})[['url', 'label']].dropna()

# PhiUSIIL encodes: 1 = Legitimate, 0 = Phishing → invert to standard ML convention
if df['label'].dtype in ['int64', 'float64']:
    if 'urllength' in cols_lower and 'tldlegitimateprob' in cols_lower:
        print('ℹ️  PhiUSIIL format detected — inverting labels (1→Legit becomes 0=Legit)')
        df['label'] = 1 - df['label'].astype(int)
elif df['label'].dtype == object:
    df['label'] = df['label'].apply(
        lambda x: 0 if str(x).lower() in ['benign', 'good', 'legitimate', '0'] else 1
    )

df['url'] = df['url'].astype(str)
print(f'\n📦 Clean dataset: {len(df):,} rows')
print(df['label'].value_counts().rename({0: 'Legitimate (0)', 1: 'Phishing (1)'}))

In [ ]:
# ─── Class Distribution Plot ────────────────────────────────────────────────────
counts = df['label'].value_counts().sort_index()
labels_str = ['Legitimate', 'Phishing']
colors = [LEGIT_COLOR, PHISH_COLOR]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dataset Class Distribution', fontsize=16, fontweight='bold', color='#e6edf3', y=1.01)

# Bar chart
bars = axes[0].bar(labels_str, counts.values, color=colors, edgecolor='#30363d', linewidth=1.2)
axes[0].set_title('Sample Counts', fontsize=13)
axes[0].set_ylabel('Number of URLs')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].grid(axis='y')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'{val:,}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels_str, colors=colors,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2},
    textprops={'color': '#e6edf3', 'fontsize': 12}
)
for at in autotexts:
    at.set_fontsize(13)
    at.set_fontweight('bold')
axes[1].set_title('Class Proportion', fontsize=13)
axes[1].set_facecolor('#161b22')

plt.tight_layout()
plt.show()

In [ ]:
# ─── URL Length Analysis ────────────────────────────────────────────────────────
df['url_length'] = df['url'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('URL Length Distribution by Class', fontsize=15, fontweight='bold', color='#e6edf3')

for label_val, label_name, color in [(0, 'Legitimate', LEGIT_COLOR), (1, 'Phishing', PHISH_COLOR)]:
    data = df[df['label'] == label_val]['url_length']
    axes[0].hist(data, bins=60, alpha=0.7, label=label_name, color=color, edgecolor='none')

axes[0].set_xlabel('URL Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Histogram')
axes[0].legend()
axes[0].grid(axis='y')

data_to_plot = [df[df['label'] == 0]['url_length'], df[df['label'] == 1]['url_length']]
bp = axes[1].boxplot(data_to_plot, labels=['Legitimate', 'Phishing'],
                     patch_artist=True, notch=True,
                     medianprops={'color': '#f0f6fc', 'linewidth': 2})
for patch, color in zip(bp['boxes'], [LEGIT_COLOR, PHISH_COLOR]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel('URL Length')
axes[1].set_title('Box Plot')
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

print('URL Length Stats:')
print(df.groupby('label')['url_length'].describe().rename(index={0:'Legitimate',1:'Phishing'}).round(1).to_string())

## 2. TF-IDF Feature Analysis

In [ ]:
# ─── TF-IDF Vectoriser ──────────────────────────────────────────────────────────
print('Building TF-IDF vocabulary on full dataset...')
tfidf_preview = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=5000)
X_tfidf = tfidf_preview.fit_transform(df['url'])

feature_names = tfidf_preview.get_feature_names_out()
print(f'✅ Vocabulary size: {len(feature_names):,} character n-grams')

# Top TF-IDF mean scores per class
X_dense = pd.DataFrame.sparse.from_spmatrix(X_tfidf, columns=feature_names)
X_dense['label'] = df['label'].values

phish_means = X_dense[X_dense['label'] == 1].drop('label', axis=1).mean()
legit_means  = X_dense[X_dense['label'] == 0].drop('label', axis=1).mean()

top_phish = phish_means.nlargest(20)
top_legit  = legit_means.nlargest(20)
print('\nTop 5 phishing n-grams:', list(top_phish.head(5).index))
print('Top 5 legit n-grams:   ', list(top_legit.head(5).index))

In [ ]:
# ─── TF-IDF Top Features Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('TF-IDF: Top 20 Character N-Grams by Class\n(mean TF-IDF weight across all URLs)',
             fontsize=14, fontweight='bold', color='#e6edf3')

for ax, data, title, color in [
    (axes[0], top_phish, '🔴 Phishing URLs', PHISH_COLOR),
    (axes[1], top_legit,  '🟢 Legitimate URLs', LEGIT_COLOR)
]:
    bars = ax.barh(range(len(data)), data.values, color=color, alpha=0.85, edgecolor='none')
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels([repr(x) for x in data.index], fontsize=9.5, fontfamily='monospace')
    ax.set_xlabel('Mean TF-IDF Weight')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x')

plt.tight_layout()
plt.show()

## 3. Model Training

In [ ]:
# ─── Train / Test Split ─────────────────────────────────────────────────────────
X = df['url'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} samples')
print(f'Test : {len(X_test):,} samples')
print(f'Train phishing rate: {y_train.mean():.2%}')
print(f'Test  phishing rate: {y_test.mean():.2%}')

In [ ]:
# ─── Train Model ────────────────────────────────────────────────────────────────
print('⏳ Training TF-IDF + Random Forest pipeline…')

model = make_pipeline(
    TfidfVectorizer(analyzer='char', ngram_range=(3, 5)),
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
)

model.fit(X_train, y_train)
print('✅ Training complete!')

## 4. Evaluation Metrics

In [ ]:
# ─── Core Metrics ───────────────────────────────────────────────────────────────
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

print('=' * 42)
print(f'  Accuracy  : {acc:.4f}  ({acc:.1%})')
print(f'  Precision : {prec:.4f}  ({prec:.1%})')
print(f'  Recall    : {rec:.4f}  ({rec:.1%})')
print(f'  F1-Score  : {f1:.4f}  ({f1:.1%})')
print(f'  ROC-AUC   : {roc_auc:.4f}')
print('=' * 42)
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Phishing']))

In [ ]:
# ─── Metrics Dashboard ──────────────────────────────────────────────────────────
metric_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metric_values = [acc, prec, rec, f1, roc_auc]
bar_colors = [ACCENT_COLOR if v >= 0.90 else '#d29922' for v in metric_values]

fig, ax = plt.subplots(figsize=(11, 4.5))
bars = ax.bar(metric_names, metric_values, color=bar_colors, edgecolor='#30363d', linewidth=1.1, width=0.55)
ax.set_ylim(0, 1.10)
ax.axhline(0.90, color='#3fb950', linestyle='--', linewidth=1, label='90% threshold')
ax.set_ylabel('Score')
ax.set_title('Model Performance Metrics', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='y')

for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.012,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Confusion Matrix

In [ ]:
# ─── Confusion Matrix ───────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('Confusion Matrix', fontsize=15, fontweight='bold', color='#e6edf3')

# Raw counts
cmap_phish = sns.diverging_palette(145, 10, as_cmap=True)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'],
            linewidths=1, linecolor='#0d1117',
            annot_kws={'size': 16, 'weight': 'bold'},
            ax=axes[0])
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
axes[0].set_title('Raw Counts')

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='RdYlGn',
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'],
            linewidths=1, linecolor='#0d1117',
            annot_kws={'size': 14, 'weight': 'bold'},
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)
axes[1].set_title('Normalised (per True Class)')

tn, fp, fn, tp = cm.ravel()
print(f'True  Negatives (correctly caught legitimate): {tn:,}')
print(f'False Positives (legitimate flagged as phish): {fp:,}')
print(f'False Negatives (phishing missed by model)  : {fn:,}')
print(f'True  Positives (phishing correctly caught) : {tp:,}')

plt.tight_layout()
plt.show()

## 6. ROC Curve & AUC

In [ ]:
# ─── ROC Curve ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

ax.plot(fpr, tpr, color=ACCENT_COLOR, lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#8b949e', lw=1.2, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.08, color=ACCENT_COLOR)

# Mark a few operating points
for threshold_label, threshold_fpr in [('FPR=1%', 0.01), ('FPR=5%', 0.05), ('FPR=10%', 0.10)]:
    idx = np.searchsorted(fpr, threshold_fpr)
    if idx < len(tpr):
        ax.scatter(fpr[idx], tpr[idx], s=80, zorder=5, color='#f0a500')
        ax.annotate(f'{threshold_label}\nTPR={tpr[idx]:.2f}',
                    (fpr[idx], tpr[idx]), xytext=(fpr[idx]+0.05, tpr[idx]-0.06),
                    fontsize=9, color='#f0a500',
                    arrowprops=dict(arrowstyle='->', color='#f0a500', lw=1.2))

ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Phish-Guard ML Model', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True)

plt.tight_layout()
plt.show()

## 7. Feature Importance (Random Forest)

In [ ]:
# ─── Feature Importances ────────────────────────────────────────────────────────
vectorizer = model.named_steps['tfidfvectorizer']
classifier = model.named_steps['randomforestclassifier']

feature_names_model = vectorizer.get_feature_names_out()
importances = classifier.feature_importances_

feat_df = pd.DataFrame({'ngram': feature_names_model, 'importance': importances})
top_n = feat_df.nlargest(30, 'importance')

fig, ax = plt.subplots(figsize=(12, 8))
colors_fi = [PHISH_COLOR if i < 15 else LEGIT_COLOR for i in range(len(top_n))]
bars = ax.barh(range(len(top_n)), top_n['importance'].values,
               color=colors_fi, alpha=0.85, edgecolor='none')
ax.set_yticks(range(len(top_n)))
ax.set_yticklabels([repr(x) for x in top_n['ngram']], fontsize=9, fontfamily='monospace')
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Top 30 TF-IDF Character N-Grams by Random Forest Importance',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PHISH_COLOR, alpha=0.85, label='Higher Phishing Signal'),
                   Patch(facecolor=LEGIT_COLOR, alpha=0.85, label='Higher Legit Signal')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 8. Cross-Validation

In [ ]:
# ─── 5-Fold Stratified Cross-Validation ─────────────────────────────────────────
print('⏳ Running 5-fold cross-validation (this may take a few minutes)…')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
for metric, scoring in [('Accuracy','accuracy'), ('F1','f1'), ('ROC-AUC','roc_auc')]:
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[metric] = scores
    print(f'  {metric:8s}: {scores.mean():.4f} ± {scores.std():.4f}  |  folds: {np.round(scores, 4)}')

print('\n✅ Cross-validation complete!')

In [ ]:
# ─── CV Results Plot ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('5-Fold Stratified Cross-Validation Results', fontsize=14, fontweight='bold', color='#e6edf3')

metric_colors = [ACCENT_COLOR, '#d2a8ff', '#f0a500']
for ax, (metric, scores), color in zip(axes, cv_results.items(), metric_colors):
    folds = [f'Fold {i+1}' for i in range(len(scores))]
    bars = ax.bar(folds, scores, color=color, alpha=0.85, edgecolor='#30363d')
    ax.axhline(scores.mean(), color='#fff', linestyle='--', linewidth=1.5,
               label=f'Mean: {scores.mean():.4f}')
    ax.set_ylim(max(0, scores.min() - 0.02), min(1.05, scores.max() + 0.04))
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend(fontsize=10)
    ax.grid(axis='y')
    for bar, val in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9.5)

plt.tight_layout()
plt.show()

## 9. Prediction Score Distribution

In [ ]:
# ─── Probability Score Distribution ─────────────────────────────────────────────
proba_legit = y_proba[y_test == 0]
proba_phish = y_proba[y_test == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Predicted Phishing Probability Distribution', fontsize=14, fontweight='bold')

# Overlapping histogram
axes[0].hist(proba_legit, bins=50, alpha=0.65, color=LEGIT_COLOR, label='Legitimate', density=True)
axes[0].hist(proba_phish, bins=50, alpha=0.65, color=PHISH_COLOR, label='Phishing',   density=True)
axes[0].axvline(0.5, color='#f0f6fc', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')
axes[0].set_xlabel('Predicted Probability (Phishing)')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Histogram')
axes[0].legend()
axes[0].grid(axis='y')

# Violin
data_violin = [proba_legit, proba_phish]
vp = axes[1].violinplot(data_violin, positions=[1, 2], showmedians=True)
for body, color in zip(vp['bodies'], [LEGIT_COLOR, PHISH_COLOR]):
    body.set_facecolor(color)
    body.set_alpha(0.75)
vp['cmedians'].set_color('#f0f6fc')
vp['cmedians'].set_linewidth(2)
axes[1].set_xticks([1, 2])
axes[1].set_xticklabels(['Legitimate', 'Phishing'])
axes[1].set_ylabel('Predicted Probability (Phishing)')
axes[1].set_title('Violin Plot')
axes[1].axhline(0.5, color='#f0f6fc', linestyle='--', linewidth=1.2, label='Decision boundary')
axes[1].legend()
axes[1].grid(axis='y')

plt.tight_layout()
plt.show()

## 10. Live Prediction Examples

In [ ]:
# ─── Live URL Prediction Demo ────────────────────────────────────────────────────
test_urls = [
    # Legitimate
    'https://www.coinbase.com/signin',
    'https://www.google.com/search?q=crypto',
    'https://github.com/shri-915/phish-guard',
    # Phishing
    'http://coinbase-support-wallet.xyz/login',
    'http://secure-coinbase-verify.net/kyc-update',
    'http://coinbase-airdrop-claim.com/free-eth',
    # Edge cases
    'https://coinbase.com.phish-site.ru/login',
    'https://help.coinbase-pro-support.com',
]

print(f'{"URL":<55} {"Phish Score":>12}  {"Decision"}')
print('─' * 85)
for url in test_urls:
    score = model.predict_proba([url])[0][1]
    decision = '🔴 PHISHING' if score > 0.65 else ('⚠️  CAUTION' if score > 0.40 else '🟢 SAFE')
    print(f'{url[:54]:<55} {score:>11.4f}  {decision}')

## 11. Summary Report

In [ ]:
# ─── Final Summary ───────────────────────────────────────────────────────────────
print('=' * 52)
print('         🛡️  PHISH-GUARD — MODEL SUMMARY')
print('=' * 52)
print(f'  Dataset         : PhiUSIIL Phishing URL Dataset')
print(f'  Total Samples   : {len(df):>10,}')
print(f'  Training Set    : {len(X_train):>10,}')
print(f'  Test Set        : {len(X_test):>10,}')
print(f'  Algorithm       : Random Forest (100 trees)')
print(f'  Features        : TF-IDF char n-grams (3–5)')
print('─' * 52)
print(f'  Accuracy        : {acc:>10.4f}  ({acc:.1%})')
print(f'  Precision       : {prec:>10.4f}  ({prec:.1%})')
print(f'  Recall          : {rec:>10.4f}  ({rec:.1%})')
print(f'  F1-Score        : {f1:>10.4f}  ({f1:.1%})')
print(f'  ROC-AUC         : {roc_auc:>10.4f}')
print('─' * 52)
cv_acc  = cv_results['Accuracy']
cv_f1   = cv_results['F1']
cv_roc  = cv_results['ROC-AUC']
print(f'  CV Accuracy     : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')
print(f'  CV F1           : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  CV ROC-AUC      : {cv_roc.mean():.4f} ± {cv_roc.std():.4f}')
print('=' * 52)